In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent


TRAIN_CHECKPOINT_PATH = (
    PROJECT_DIR
    / "data"
    / "GSE25055_pre_lasso.joblib"
)

EXTERNAL_CHECKPOINT_PATH = (
    PROJECT_DIR
    / "data"
    / "GSE25065_pre_lasso.joblib"
)


RESULTS_DIR = PROJECT_DIR / "results" / "lasso_balanced_rf"

EXPERIMENT_PATH = RESULTS_DIR / "experiment_results.joblib"



required_files = [
    TRAIN_CHECKPOINT_PATH,
    EXTERNAL_CHECKPOINT_PATH,
    EXPERIMENT_PATH
]

for file_path in required_files:
    print(
        file_path.name,
        "exists:",
        file_path.exists()
    )

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path.resolve()}"
        )


training_checkpoint = joblib.load(
    TRAIN_CHECKPOINT_PATH
)

external_checkpoint = joblib.load(
    EXTERNAL_CHECKPOINT_PATH
)

experiment_results = joblib.load(
    EXPERIMENT_PATH
)


X_train = training_checkpoint["X"]
y_train = training_checkpoint["y"]

X_external = external_checkpoint["X"]
y_external = external_checkpoint["y"]

oof_predictions = experiment_results[
    "oof_predictions"
]


print("\nGSE25055:")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nGSE25065:")
print("X:", X_external.shape)
print("y:", y_external.shape)

print("\nOOF predictions:")
print(oof_predictions.shape)

GSE25055_pre_lasso.joblib exists: True
GSE25065_pre_lasso.joblib exists: True
experiment_results.joblib exists: True

GSE25055:
X: (306, 22283)
y: (306,)

GSE25065:
X: (182, 22283)
y: (182,)

OOF predictions:
(306, 6)


In [2]:
print("OOF columns:")
print(oof_predictions.columns.tolist())

print("\nMissing values:")
print(oof_predictions.isna().sum())

display(oof_predictions.head())

OOF columns:
['patient', 'actual_class', 'custom_prediction', 'custom_probability', 'sklearn_prediction', 'sklearn_probability']

Missing values:
patient                0
actual_class           0
custom_prediction      0
custom_probability     0
sklearn_prediction     0
sklearn_probability    0
dtype: int64


,patient,actual_class,custom_prediction,custom_probability,sklearn_prediction,sklearn_probability
0,GSM615096,0,0,0.000000,0,0.039714
1,GSM615097,0,0,0.192274,0,0.207505
2,GSM615098,0,1,0.687641,1,0.895085
3,GSM615099,1,1,0.653574,1,0.720642
4,GSM615100,0,0,0.127152,0,0.180617


In [3]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)


# Подравняване и проверка на OOF резултатите
oof_aligned = (
    oof_predictions
    .set_index("patient")
    .reindex(y_train.index)
)

assert not oof_aligned.isna().any().any()
assert np.array_equal(
    oof_aligned["actual_class"].to_numpy(dtype=int),
    y_train.to_numpy(dtype=int)
)

y_oof = oof_aligned["actual_class"].to_numpy(dtype=int)


def evaluate_threshold(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "threshold": threshold,
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            predictions
        ),
        "sensitivity": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "accuracy": accuracy_score(
            y_true,
            predictions
        )
    }


threshold_values = np.round(
    np.arange(0.05, 0.96, 0.01),
    2
)

threshold_records = []

for model_name, probability_column in {
    "Custom Random Forest": "custom_probability",
    "Sklearn Random Forest": "sklearn_probability"
}.items():

    probabilities = oof_aligned[
        probability_column
    ].to_numpy(dtype=float)

    for threshold in threshold_values:
        result = evaluate_threshold(
            y_oof,
            probabilities,
            threshold
        )

        threshold_records.append({
            "model": model_name,
            **result
        })


threshold_results = pd.DataFrame(threshold_records)

best_threshold_rows = []

for model_name, group in threshold_results.groupby("model"):
    group = group.copy()

    # При равен резултат избираме прага,
    # който е най-близо до стандартния 0.50.
    group["distance_from_050"] = (
        group["threshold"] - 0.50
    ).abs()

    best_row = (
        group
        .sort_values(
            by=[
                "balanced_accuracy",
                "distance_from_050"
            ],
            ascending=[False, True]
        )
        .iloc[0]
    )

    best_threshold_rows.append(best_row)


best_thresholds = pd.DataFrame(
    best_threshold_rows
).drop(columns="distance_from_050")

LOCKED_THRESHOLDS = {
    row["model"]: float(row["threshold"])
    for _, row in best_thresholds.iterrows()
}

print("Locked thresholds selected only from GSE25055 OOF predictions:")
print(LOCKED_THRESHOLDS)

display(
    best_thresholds[
        [
            "model",
            "threshold",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "precision",
            "f1",
            "accuracy"
        ]
    ].round(3)
)

Locked thresholds selected only from GSE25055 OOF predictions:
{'Custom Random Forest': 0.21, 'Sklearn Random Forest': 0.35}


,model,threshold,balanced_accuracy,sensitivity,specificity,precision,f1,accuracy
16,Custom Random Forest,0.21,0.752,0.789,0.715,0.388,0.520,0.729
121,Sklearn Random Forest,0.35,0.714,0.702,0.727,0.370,0.485,0.722


In [4]:
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


FINAL_RANDOM_STATE = 42


def create_lasso_pipeline():
    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "lasso",
            LogisticRegression(
                l1_ratio=1.0,
                solver="liblinear",
                class_weight="balanced",
                max_iter=10000,
                random_state=FINAL_RANDOM_STATE
            )
        )
    ])


parameter_grid = {
    "lasso__C": [
        0.001,
        0.003,
        0.01,
        0.03,
        0.1,
        0.3
    ]
}


final_inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=FINAL_RANDOM_STATE
)


final_lasso_search = GridSearchCV(
    estimator=create_lasso_pipeline(),
    param_grid=parameter_grid,
    scoring="average_precision",
    cv=final_inner_cv,
    refit=True,
    n_jobs=-1
)


final_lasso_search.fit(
    X_train,
    y_train
)


final_coefficients = (
    final_lasso_search
    .best_estimator_
    .named_steps["lasso"]
    .coef_
    .ravel()
)

final_selected_mask = (
    np.abs(final_coefficients) > 1e-10
)

final_selected_probes = (
    X_train.columns[final_selected_mask]
    .tolist()
)


assert len(final_selected_probes) > 0

missing_external_probes = (
    set(final_selected_probes)
    - set(X_external.columns)
)

assert len(missing_external_probes) == 0


print(
    "Final selected C:",
    final_lasso_search.best_params_["lasso__C"]
)

print(
    "Mean inner-CV average precision:",
    round(final_lasso_search.best_score_, 3)
)

print(
    "Final selected probes:",
    len(final_selected_probes)
)

print(
    "Missing selected probes from GSE25065:",
    len(missing_external_probes)
)

Final selected C: 0.03
Mean inner-CV average precision: 0.531
Final selected probes: 15
Missing selected probes from GSE25065: 0


In [6]:
import time

from sklearn.ensemble import RandomForestClassifier
from pathlib import Path
import sys
import time

from sklearn.ensemble import RandomForestClassifier


PROJECT_DIR = Path(r"D:\diplom-project")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

CUSTOM_RF_PATH = (
    PROJECT_DIR
    / "src"
    / "models"
    / "custom_random_forest.py"
)

print("Project directory:", PROJECT_DIR)
print("Custom RF file exists:", CUSTOM_RF_PATH.exists())

if not CUSTOM_RF_PATH.exists():
    raise FileNotFoundError(
        f"Custom Random Forest not found: {CUSTOM_RF_PATH}"
    )


from src.models.custom_random_forest import build_forest

FINAL_RF_CONFIG = {
    "number_of_trees": 30,
    "max_depth": 6,
    "min_samples_split": 5,
    "max_features": None,
    "class_weight": "balanced",
    "random_state": 42
}


X_train_selected = (
    X_train[final_selected_probes]
    .to_numpy(dtype=float)
)

y_train_array = y_train.to_numpy(dtype=int)


print("Training matrix:", X_train_selected.shape)
print("Training Custom Random Forest...")


custom_start = time.time()

final_custom_forest = build_forest(
    X_train_selected,
    y_train_array,
    number_of_trees=FINAL_RF_CONFIG["number_of_trees"],
    max_depth=FINAL_RF_CONFIG["max_depth"],
    min_samples_split=FINAL_RF_CONFIG["min_samples_split"],
    max_features=FINAL_RF_CONFIG["max_features"],
    class_weight=FINAL_RF_CONFIG["class_weight"],
    random_state=FINAL_RF_CONFIG["random_state"]
)

custom_training_time = time.time() - custom_start


print("\nTraining Sklearn Random Forest...")

sklearn_start = time.time()

final_sklearn_forest = RandomForestClassifier(
    n_estimators=30,
    max_depth=6,
    min_samples_split=5,
    max_features="sqrt",
    bootstrap=True,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

final_sklearn_forest.fit(
    X_train_selected,
    y_train_array
)

sklearn_training_time = time.time() - sklearn_start


assert len(final_custom_forest) == 30
assert len(final_sklearn_forest.estimators_) == 30


print("\nFinal models trained successfully.")
print(
    "Custom trees:",
    len(final_custom_forest),
    "| time:",
    round(custom_training_time, 2),
    "seconds"
)
print(
    "Sklearn trees:",
    len(final_sklearn_forest.estimators_),
    "| time:",
    round(sklearn_training_time, 2),
    "seconds"
)

Project directory: D:\diplom-project
Custom RF file exists: True
Training matrix: (306, 15)
Training Custom Random Forest...
Building tree 1 of 30
Building tree 2 of 30
Building tree 3 of 30
Building tree 4 of 30
Building tree 5 of 30
Building tree 6 of 30
Building tree 7 of 30
Building tree 8 of 30
Building tree 9 of 30
Building tree 10 of 30
Building tree 11 of 30
Building tree 12 of 30
Building tree 13 of 30
Building tree 14 of 30
Building tree 15 of 30
Building tree 16 of 30
Building tree 17 of 30
Building tree 18 of 30
Building tree 19 of 30
Building tree 20 of 30
Building tree 21 of 30
Building tree 22 of 30
Building tree 23 of 30
Building tree 24 of 30
Building tree 25 of 30
Building tree 26 of 30
Building tree 27 of 30
Building tree 28 of 30
Building tree 29 of 30
Building tree 30 of 30

Training Sklearn Random Forest...

Final models trained successfully.
Custom trees: 30 | time: 10.14 seconds
Sklearn trees: 30 | time: 0.07 seconds


In [7]:
from datetime import datetime, timezone
from pathlib import Path

import joblib
import sklearn


FINAL_MODEL_DIR = (
    PROJECT_DIR
    / "results"
    / "final_model_gse25055"
)

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_MODEL_PATH = (
    FINAL_MODEL_DIR
    / "final_model_package.joblib"
)


final_model_package = {
    "format_version": "1.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "training_dataset": "GSE25055",
    "external_dataset": "GSE25065",

    "label_mapping": {
        "RD": 0,
        "pCR": 1
    },

    "training_samples": len(X_train),
    "original_feature_count": X_train.shape[1],

    "lasso_best_C": (
        final_lasso_search.best_params_["lasso__C"]
    ),
    "lasso_inner_cv_average_precision": (
        final_lasso_search.best_score_
    ),
    "lasso_pipeline": (
        final_lasso_search.best_estimator_
    ),

    "selected_probes": final_selected_probes,
    "selected_probe_count": len(final_selected_probes),

    "custom_random_forest": final_custom_forest,
    "sklearn_random_forest": final_sklearn_forest,

    "custom_threshold": LOCKED_THRESHOLDS[
        "Custom Random Forest"
    ],
    "sklearn_threshold": LOCKED_THRESHOLDS[
        "Sklearn Random Forest"
    ],

    "custom_rf_config": FINAL_RF_CONFIG,

    "sklearn_rf_config": {
        "n_estimators": 30,
        "max_depth": 6,
        "min_samples_split": 5,
        "max_features": "sqrt",
        "bootstrap": True,
        "class_weight": "balanced",
        "random_state": 42
    },

    "custom_training_time_seconds": (
        custom_training_time
    ),
    "sklearn_training_time_seconds": (
        sklearn_training_time
    ),

    "sklearn_version": sklearn.__version__
}


joblib.dump(
    final_model_package,
    FINAL_MODEL_PATH,
    compress=3
)


# Отделен CSV за по-лесно разглеждане
selected_probes_path = (
    FINAL_MODEL_DIR
    / "final_selected_probes.csv"
)

pd.DataFrame({
    "probe_id": final_selected_probes,
    "lasso_coefficient": (
        final_coefficients[final_selected_mask]
    )
}).to_csv(
    selected_probes_path,
    index=False
)


# Проверка чрез повторно зареждане
saved_package = joblib.load(
    FINAL_MODEL_PATH
)

assert saved_package["training_dataset"] == "GSE25055"
assert saved_package["selected_probe_count"] == 15
assert len(saved_package["custom_random_forest"]) == 30
assert (
    len(
        saved_package[
            "sklearn_random_forest"
        ].estimators_
    )
    == 30
)


print("Final model package saved:")
print(FINAL_MODEL_PATH.resolve())

print("\nSelected probes saved:")
print(selected_probes_path.resolve())

print("\nLocked configuration:")
print(
    "LASSO C:",
    saved_package["lasso_best_C"]
)
print(
    "Selected probes:",
    saved_package["selected_probe_count"]
)
print(
    "Custom threshold:",
    saved_package["custom_threshold"]
)
print(
    "Sklearn threshold:",
    saved_package["sklearn_threshold"]
)

Final model package saved:
D:\diplom-project\results\final_model_gse25055\final_model_package.joblib

Selected probes saved:
D:\diplom-project\results\final_model_gse25055\final_selected_probes.csv

Locked configuration:
LASSO C: 0.03
Selected probes: 15
Custom threshold: 0.21
Sklearn threshold: 0.35


In [8]:
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score
)

from src.models.custom_random_forest import predict_forest


# Зареждаме заключения пакет от диска
locked_package = joblib.load(
    FINAL_MODEL_PATH
)

locked_probes = locked_package[
    "selected_probes"
]

custom_threshold = locked_package[
    "custom_threshold"
]

sklearn_threshold = locked_package[
    "sklearn_threshold"
]


# Проверка за съвместимост
missing_probes = [
    probe
    for probe in locked_probes
    if probe not in X_external.columns
]

if missing_probes:
    raise ValueError(
        f"Missing probes in GSE25065: {missing_probes}"
    )

X_external_selected = (
    X_external
    .loc[:, locked_probes]
    .to_numpy(dtype=float)
)

y_external_array = (
    y_external.to_numpy(dtype=int)
)

assert X_external_selected.shape == (
    len(y_external),
    len(locked_probes)
)

print(
    "External evaluation matrix:",
    X_external_selected.shape
)
print(
    "Custom threshold:",
    custom_threshold
)
print(
    "Sklearn threshold:",
    sklearn_threshold
)


# Custom Random Forest — само predict
custom_predictions, custom_probabilities = predict_forest(
    locked_package["custom_random_forest"],
    X_external_selected,
    classification_threshold=custom_threshold
)


# Sklearn Random Forest — само predict
sklearn_probabilities = (
    locked_package["sklearn_random_forest"]
    .predict_proba(X_external_selected)[:, 1]
)

sklearn_predictions = (
    sklearn_probabilities >= sklearn_threshold
).astype(int)


def calculate_external_metrics(
    model_name,
    y_true,
    predictions,
    probabilities,
    threshold
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": model_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),
        "pr_auc": average_precision_score(
            y_true,
            probabilities
        ),
        "accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                predictions
            )
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "sensitivity": (
            tp / (tp + fn)
            if (tp + fn) > 0
            else np.nan
        ),
        "specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


external_metrics = pd.DataFrame([
    calculate_external_metrics(
        "Custom Random Forest",
        y_external_array,
        custom_predictions,
        custom_probabilities,
        custom_threshold
    ),
    calculate_external_metrics(
        "Sklearn Random Forest",
        y_external_array,
        sklearn_predictions,
        sklearn_probabilities,
        sklearn_threshold
    )
])


external_predictions = pd.DataFrame({
    "patient": y_external.index,
    "actual_class": y_external_array,
    "custom_probability": custom_probabilities,
    "custom_prediction": custom_predictions,
    "sklearn_probability": sklearn_probabilities,
    "sklearn_prediction": sklearn_predictions
})


# Незабавно записване на окончателните резултати
EXTERNAL_RESULTS_DIR = (
    PROJECT_DIR
    / "results"
    / "final_external_validation_gse25065"
)

EXTERNAL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

external_metrics_path = (
    EXTERNAL_RESULTS_DIR
    / "external_metrics.csv"
)

external_predictions_path = (
    EXTERNAL_RESULTS_DIR
    / "external_predictions.csv"
)

external_bundle_path = (
    EXTERNAL_RESULTS_DIR
    / "external_validation_results.joblib"
)


external_metrics.to_csv(
    external_metrics_path,
    index=False
)

external_predictions.to_csv(
    external_predictions_path,
    index=False
)

joblib.dump(
    {
        "evaluated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "training_dataset": "GSE25055",
        "external_dataset": "GSE25065",
        "model_package_path": str(
            FINAL_MODEL_PATH.resolve()
        ),
        "selected_probes": locked_probes,
        "metrics": external_metrics,
        "predictions": external_predictions
    },
    external_bundle_path,
    compress=3
)


print("\nFINAL EXTERNAL VALIDATION — GSE25065")

display(
    external_metrics.round(3)
)

print("Metrics saved:", external_metrics_path.resolve())
print("Predictions saved:", external_predictions_path.resolve())
print("Result bundle saved:", external_bundle_path.resolve())

External evaluation matrix: (182, 15)
Custom threshold: 0.21
Sklearn threshold: 0.35

FINAL EXTERNAL VALIDATION — GSE25065


,model,threshold,roc_auc,pr_auc,accuracy,balanced_accuracy,f1,precision,sensitivity,specificity,tn,fp,fn,tp
0,Custom Random Forest,0.21,0.695,0.402,0.659,0.629,0.436,0.353,0.571,0.686,96,44,18,24
1,Sklearn Random Forest,0.35,0.711,0.407,0.676,0.656,0.468,0.377,0.619,0.693,97,43,16,26


Metrics saved: D:\diplom-project\results\final_external_validation_gse25065\external_metrics.csv
Predictions saved: D:\diplom-project\results\final_external_validation_gse25065\external_predictions.csv
Result bundle saved: D:\diplom-project\results\final_external_validation_gse25065\external_validation_results.joblib
